# Qwen3-VL: Извлечение данных из PDF с помощью Vision Language Model

Эта модель "видит" страницу PDF как изображение и извлекает:
- Текст
- Данные из таблиц
- Информацию с графиков и диаграмм
- Контекст изображений

In [ ]:
# Установка зависимостей
!pip install -q git+https://github.com/huggingface/transformers
!pip install -q accelerate qwen-vl-utils pdf2image pillow
!apt-get install -q poppler-utils  # для конвертации PDF в изображения

In [ ]:
# Загрузка PDF файлов в Colab
from google.colab import files

print("Загрузите PDF файлы:")
uploaded = files.upload()

In [ ]:
# Загрузка модели Qwen3-VL
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

model_name = "Qwen/Qwen3-VL-30B-A3B-Instruct"

model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

processor = AutoProcessor.from_pretrained(model_name)
print("Модель загружена!")

In [ ]:
# Функция конвертации PDF страницы в изображение
from pdf2image import convert_from_path
from PIL import Image

def pdf_page_to_image(pdf_path, page_number):
    """
    Конвертирует страницу PDF в изображение
    page_number: нумерация с 1
    """
    images = convert_from_path(
        pdf_path, 
        first_page=page_number, 
        last_page=page_number,
        dpi=200  # хорошее качество для OCR
    )
    return images[0] if images else None

In [ ]:
# Функция извлечения данных с помощью Qwen VL
def extract_data_from_image(image, prompt):
    """
    Отправляет изображение в Qwen VL и получает извлеченные данные
    """
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]
    
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)
    
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs, 
            max_new_tokens=2048,
            do_sample=False
        )
    
    generated_ids_trimmed = [
        out_ids[len(in_ids):] 
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    
    output_text = processor.batch_decode(
        generated_ids_trimmed, 
        skip_special_tokens=True
    )[0]
    
    return output_text

In [ ]:
# Пример 1: Простой документ (страница 3)
pdf_path = "NIRM03_SelfLearning_ASR_Kazakh_Updated.pdf"
page_num = 3

# Конвертируем страницу в изображение
image = pdf_page_to_image(pdf_path, page_num)
display(image)  # показываем страницу

# Промпт для извлечения данных
prompt = """Извлеки весь текст с этой страницы документа. 
Сохрани структуру и форматирование. 
Если есть заголовки, списки или таблицы - отформатируй их соответственно."""

result = extract_data_from_image(image, prompt)

print("=" * 50)
print("QWEN VL - ПРОСТОЙ ТЕКСТ (страница 3)")
print("=" * 50)
print(result)

In [ ]:
# Пример 2: Сложная таблица из финансовой отчетности (страница 23)
pdf_path = "09_2025_Consolidated Financial statements_IFRS_RUS.pdf"
page_num = 23

# Конвертируем страницу в изображение
image = pdf_page_to_image(pdf_path, page_num)
display(image)  # показываем страницу

# Промпт для извлечения таблиц и данных
prompt = """Проанализируй эту страницу финансового отчета.

1. Извлеки все таблицы в формате markdown
2. Сохрани точные числовые значения
3. Укажи заголовки столбцов и строк
4. Если есть примечания или сноски - включи их

Верни структурированные данные."""

result = extract_data_from_image(image, prompt)

print("=" * 50)
print("QWEN VL - СЛОЖНАЯ ТАБЛИЦА (страница 23)")
print("=" * 50)
print(result)
print("\n")
print("=" * 50)
print("ВЫВОД: VLM модель 'видит' страницу и понимает структуру!")
print("Таблицы, графики, диаграммы - всё извлекается с контекстом.")
print("=" * 50)

In [ ]:
# Бонус: Извлечение данных с графиков
# Если на странице есть график - VLM может его интерпретировать

def extract_chart_data(image):
    prompt = """Проанализируй изображение.
    
    Если есть графики или диаграммы:
    1. Опиши тип графика (линейный, столбчатый, круговой и т.д.)
    2. Извлеки данные с осей (названия, значения)
    3. Извлеки все числовые значения с графика
    4. Опиши тренды и ключевые точки
    
    Если есть таблицы - извлеки их в markdown формате.
    Если есть обычный текст - извлеки его полностью."""
    
    return extract_data_from_image(image, prompt)

# Пример использования:
# result = extract_chart_data(image)
# print(result)